# 04 — Flat-Foldability and the Folded State

A crease pattern is **flat-foldable** if it can be folded onto a single plane without self-intersections. Two classical local necessary conditions are:

- **Kawasaki**: at every interior vertex, the alternating sum of consecutive sector angles is zero.
- **Maekawa**: at every interior vertex, |#mountains − #valleys| = 2.

These are necessary but not sufficient — globally one must also pick a consistent face stacking order. Eucare solves the global problem as an integer linear program (`overlap.fold_complete`).

In [ ]:
import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import matplotlib.pyplot as plt
import numpy as np

import eucare as ec
from eucare import (
    conway,
    example_graphs,
    example_tilesets,
    overlap,
    plotting,
    reciprocal_figures,
    rendering,
)


def plot_g(G, ax=None, color='black', linewidth=1.0):
    """Draw the edges of a half-edge graph G on `ax` (or the current axes)."""
    if ax is None:
        ax = plt.gca()
    lines = np.array([
        [G.geometry.to_euclidean(h.orig['pos']),
         G.geometry.to_euclidean(h.dest['pos'])]
        for h in G.halfedges_representing_edges()
    ])
    plotting.plot_lines(lines, ax=ax, colors=color, linewidths=linewidth)
    plotting.set_equal_aspect(ax)
    ax.axis('off')


In [ ]:
from eucare.search_trees import face_bfs_tree
from eucare.reciprocal_figures import assign_this_way_by_face_z_order, make_SRG


def srg_pipeline(G):
    """Run the standard SRG pipeline: BFS z-order -> SRG -> recompute."""
    central = min(G.faces, key=lambda f: np.linalg.norm(f.midpoint()))
    central['z_order'] = 0
    for orig, dest in face_bfs_tree(central):
        dest['z_order'] = orig['z_order'] + 1
    assign_this_way_by_face_z_order(G)
    SRG = make_SRG(G)
    SRG.recompute_lengths_and_angles()
    return SRG


## Build a CP and check Kawasaki at every interior vertex

In [ ]:
from eucare.reciprocal_figures import kawasaki_sum

G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=2)
G.recompute_lengths_and_angles()
SRG = srg_pipeline(G)

interior = [v for v in SRG.vertices if not v.on_border()]
residuals = [abs(kawasaki_sum(v)) for v in interior]
print(f'{len(interior)} interior vertices')
print(f'max |Kawasaki residual| = {max(residuals):.2e}')


## Solve the folded face order

`fold_complete` returns mountain/valley crease assignments and writes a stacking-order attribute on each face. It uses the PuLP modeller; CBC is the default solver, CPLEX is used automatically if available.

In [ ]:
result = overlap.fold_complete(SRG, overlap_eps=1e-8)
CP = result['CP']
n_mountain = sum(1 for h in CP.halfedges if h.attributes.get(overlap.CREASE_ASSIGNMENT) == overlap.MOUNTAIN)
n_valley   = sum(1 for h in CP.halfedges if h.attributes.get(overlap.CREASE_ASSIGNMENT) == overlap.VALLEY)
# halfedges count both directions, so divide by 2
print(f'mountain creases: {n_mountain // 2}, valley creases: {n_valley // 2}')


## Visualize the M/V assignment

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for h in CP.halfedges_representing_edges():
    a = CP.geometry.to_euclidean(h.orig['pos'])
    b = CP.geometry.to_euclidean(h.dest['pos'])
    assignment = h.attributes.get(overlap.CREASE_ASSIGNMENT, 0)
    if assignment == overlap.MOUNTAIN:
        c = rendering.MOUNTAIN_COLOR
    elif assignment == overlap.VALLEY:
        c = rendering.VALLEY_COLOR
    else:
        c = rendering.FLAT_COLOR
    ax.plot([a[0], b[0]], [a[1], b[1]], color=c, linewidth=1.2)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('mountains red, valleys blue, border grey')
plt.show()


## What's next

- [`05_Conway_Plus_SRG`](05_Conway_Plus_SRG.ipynb) — design more interesting CPs by composing Conway operators before SRG.
- [`06_Export_and_3D`](06_Export_and_3D.ipynb) — save the result to SVG / FOLD / STL.